## Module 2 Homework (2026 Cohort)

In this homework, we're going to combine data from various sources to process it in Pandas and generate additional fields.

If not stated otherwise, please use the code snippets covered in the livestream to download and process the data.

> **Note on environment differences (Windows / local Jupyter / newer library versions):**
> The livestream notebook was recorded in Google Colab with a specific set of package versions. If you're running locally (especially on Windows) or with more recent `pandas`/`yfinance`/`pyarrow` versions, you may hit a few friction points that are not related to your logic:
> - **`yfinance` MultiIndex columns:** recent `yfinance` versions return `MultiIndex` columns even for single-ticker downloads, so something like `df["Close"].iloc[0]` returns a `Series` instead of a scalar, which breaks `float()` calls and arithmetic. Flatten the columns right after each download, e.g. `df.columns = df.columns.get_level_values(0)` (or `df = df.squeeze()` for single-ticker frames), before doing any further processing.
> - **`pandas.read_html()` on Windows/newer `pandas`+`lxml`:** calling `pandas.read_html(resp.text)` directly on the response text can raise an `OSError` in this combination. Wrap the HTML string in a buffer instead: `pandas.read_html(io.StringIO(resp.text))`. Apply the same fix everywhere you scrape IPO tables from HTML.
> - **`ArrowKeyError: A type extension with name pandas.period already defined`:** this can happen when calling `.to_parquet()` a second time in the same long-running kernel session — it's a known `pandas`/`pyarrow` extension-registration issue. If you hit it while re-running a cell during debugging, restart the kernel before saving again.
> - **Different IPO data source:** the livestream/lecture notebook's `get_ipos_by_year()` helper pulls data from `stockanalysis.com`, while **this homework (Q1/Q2) explicitly requires `iposcoop.com`** — a different site with different columns/data. Make sure you're scraping the URLs given in each question below, not the lecture's source.
> - **Expected row counts (Q1/Q2/Q3) can vary slightly:** the source data isn't perfectly stable — the set of withdrawn IPOs can shift a bit over time, and `yfinance` may not always return data for exactly the same number of tickers (some delisted names come and go). So you may see counts that are a little off from the numbers mentioned in the steps below (e.g., off by one or a few). Since Q3 reuses the same downloaded IPO stock data as Q2, this same variability can carry over into Q3 as well. This is expected and fine — we're asking about aggregated statistics, and the answer options are spaced far enough apart that this natural variation shouldn't change which option is correct.

### Question 1: [IPO] Withdrawn IPOs by Company Type

**What is the total withdrawn IPO value (in $ millions) for the company class with the highest total withdrawal value?**

From the Recently Filed IPO list ([iposcoop.com/ipos-recently-filed](https://www.iposcoop.com/ipos-recently-filed/)), collect and process the data to find out which company type saw the most withdrawn IPO value before Sep 11, 2026.

#### Steps:
1. **Data Loading:** Use `pandas.read_html()` with the URL above to load the IPO recently filed table. Filter the rows to keep only those where 'Expected To Trade' is 'Withdrawn'. You should identify 32 entries.
2. **Company Classification:** Create a new column called **Company Type**, categorizing company names based on patterns (order matters, assign the first matched value):
    - "Technologies" -> Technologies
    - "Acquisition Corp", "Acquisition Corporation", or "Corp" -> Acquisition Corp
    - "Inc" or "Incorporated" -> Inc.
    - "Group" -> Group
    - "Ltd" or "Limited" -> Limited
    - "Holdings" or "Holding" -> Holdings
    - Others -> Other

    **Note:** The order of the rules above is important — use the first matching rule. For example, "EUPEC International Group Ltd." will be classified as `Group` (not `Limited`), since the "Group" rule appears before the "Ltd"/"Limited" rule. Also, matches must be exact: "Xinxu Copper Industry Technology Ltd." will be classified as `Limited` (not `Technologies`), because "Technology" does not match the "Technologies" pattern.
3. **Price Parsing:** Define a new field **Avg_price** by parsing the 'Price Low' and 'Price High' fields. Create a function to extract numeric values (e.g., '$8.00' -> 8.0) and calculate the average between low and high. Handle '-' or missing values as `None`/`NaN`.
4. **Numeric Conversion:** Convert 'Shares (millions)' and 'Est \$ Vol (millions)' to numeric formats, cleaning currency symbols (\$) and commas where necessary.
5. **Value Calculation:** Create a new column **Shares_offered_value**:
    - If `Shares (millions) * Avg_price` is not null, use that value.
    - Otherwise, use the value from the `Est $ Vol (millions)` column.
6. **Aggregation:** Group by **Company Type** and calculate the sum of **Shares_offered_value**.

**Answer:** Which class had the highest total value of withdrawals, and what was that value?

In [1]:
url = "https://www.iposcoop.com/ipos-recently-filed/"

In [2]:
from datetime import datetime

import pandas as pd

In [3]:
ipo_recently_filed_table = pd.read_html(url)[0]

ipo_recently_filed_table['File Date'] = pd.to_datetime(ipo_recently_filed_table['File Date'])

ipo_recently_filed_table = ipo_recently_filed_table[(ipo_recently_filed_table['File Date'] < datetime(2026, 9, 11)) & (ipo_recently_filed_table['Expected To Trade'] == "Withdrawn")]
len(ipo_recently_filed_table)

31

In [4]:
ipo_recently_filed_table.head(1)


,File Date,Company,Symbol,Managers,Shares (millions),Price Low,Price High,Est $ Vol (millions),Expected To Trade,SCOOP Rating
38,2026-09-10,Motive Technologies (Withdrawn),MTVE,J.P.Morgan/Citigroup/Barclays/Jefferies/RBC Ca...,0.0,NaN,NaN,$100.00,Withdrawn,S/O


In [5]:
ipo_recently_filed_table['Company Type'] = [
    'Technologies' if 'Technologies' in x else (
            'Acquisition Corp' if 'Acquisition Corp' in x or 'Acquisition Corporation' in x or 'Corp' in x else (
                'Inc.' if 'Inc' in x or 'Incorporated' in x else (
                    'Group' if 'Group' in x else (
                        'Limited' if 'Ltd' in x or 'Limited' in x else (
                            'Holdings' if 'Holdings' in x or 'Holding' in x else 'Other'
                        )
                    )
                )
            )
        ) for x in ipo_recently_filed_table['Company']
]

In [6]:
ipo_recently_filed_table.head(1)

,File Date,Company,Symbol,Managers,Shares (millions),Price Low,Price High,Est $ Vol (millions),Expected To Trade,SCOOP Rating,Company Type
38,2026-09-10,Motive Technologies (Withdrawn),MTVE,J.P.Morgan/Citigroup/Barclays/Jefferies/RBC Ca...,0.0,NaN,NaN,$100.00,Withdrawn,S/O,Technologies


In [7]:
to_parse_currency_columns = ['Price Low', 'Price High', 'Est $ Vol (millions)']
for column in to_parse_currency_columns:
    ipo_recently_filed_table[column] = ipo_recently_filed_table[column].str.replace('$', '').str.replace(r'[%,]', '', regex=True).astype(float)

In [8]:
ipo_recently_filed_table.info()

<class 'pandas.DataFrame'>
Index: 31 entries, 38 to 491
Data columns (total 11 columns):
 #   Column                Non-Null Count  Dtype         
---  ------                --------------  -----         
 0   File Date             31 non-null     datetime64[us]
 1   Company               31 non-null     str           
 2   Symbol                31 non-null     str           
 3   Managers              31 non-null     str           
 4   Shares (millions)     31 non-null     float64       
 5   Price Low             25 non-null     float64       
 6   Price High            25 non-null     float64       
 7   Est $ Vol (millions)  31 non-null     float64       
 8   Expected To Trade     31 non-null     str           
 9   SCOOP Rating          31 non-null     str           
 10  Company Type          31 non-null     str           
dtypes: datetime64[us](1), float64(4), str(6)
memory usage: 5.7 KB


In [9]:
ipo_recently_filed_table['Avg_price'] = (ipo_recently_filed_table['Price Low'] + ipo_recently_filed_table['Price High'])/2

In [10]:
ipo_recently_filed_table.isna().sum()

File Date               0
Company                 0
Symbol                  0
Managers                0
Shares (millions)       0
Price Low               6
Price High              6
Est $ Vol (millions)    0
Expected To Trade       0
SCOOP Rating            0
Company Type            0
Avg_price               6
dtype: int64

In [11]:
ipo_recently_filed_table['Shares_offered_value'] = [
    x * y if y == y else z for x, y, z in zip(
        ipo_recently_filed_table['Shares (millions)'], 
        ipo_recently_filed_table['Avg_price'], 
        ipo_recently_filed_table['Est $ Vol (millions)']
    )
]

In [12]:
ipo_recently_filed_table.groupby('Company Type')['Shares_offered_value'].sum().sort_values(ascending=False)

Company Type
Acquisition Corp    499.9850
Inc.                351.0000
Holdings            311.6575
Other               290.4450
Limited             197.2500
Technologies        184.9000
Group                32.5000
Name: Shares_offered_value, dtype: float64

**Acquisition Corp is highest at about $500M (499.99)**

### Question 2: [IPO] Median Sharpe Ratio for 2025 IPOs (First 8 Months)

**What is the median Sharpe ratio (as of 11 September 2026) for companies that went public before 1 September 2025?**

The goal is to replicate the large-scale `yfinance` OHLCV data download and perform basic financial calculations on IPO stocks.

#### Steps:
1.  **Data Loading:** Download the list of 231 IPOs in 2025 from `https://www.iposcoop.com/2025-pricings/`.
2.  **Filtering:** Filter the list to keep only those IPOs with an 'Offer Date' before **1 September 2025**. Also, exclude entries with a 0% return to ensure active tickers are processed. You should see 148 stocks.
3.  **Data Download:** Use `yfinance` to download daily stock data for the filtered tickers and save it to the `stocks_df` dataframe. Make sure you can correctly process the cases when stocks are not present in Yahoo Finance (probably delisted). At this stage you should see about 134 stocks.
4.  **Feature Engineering:**
    *   Define `growth_252d` as `Close / Close.shift(252)` to represent growth after approximately one year of trading.
    *   Calculate **annualized volatility** using the specific formula: `stocks_df['volatility'] = stocks_df['Close'].rolling(30).std() * np.sqrt(252)`.
5.  **Sharpe Ratio Calculation:** Calculate the Sharpe ratio assuming a risk-free rate of **5.0%** (0.05) - it is close to the current value of the US 10Y Treasury bond yield:
    *   `stocks_df['Sharpe'] = (stocks_df['growth_252d'] - 0.05) / stocks_df['volatility']`
6.  **Final Analysis:** Filter the resulting DataFrame to keep data only for the trading day **'2026-09-11'** and compute descriptive statistics (using the `describe()` function).

#### Expected Observations:
*   Compare the median vs. mean for `growth_252d` to see if high-growth outliers are biasing the average.
*   Identify the count of stocks that successfully reached the 252-day trading milestone.
*   Do you see examples of stocks with risk-adjusted returns (Sharpe ratio) more attractive than pure growth statistics?

**Answer:** What is the median Sharpe ratio for these stocks as of September 11, 2026?

In [13]:
url = 'https://www.iposcoop.com/2025-pricings/'
ipo_2025_pricings = pd.read_html(url)[0]

ipo_2025_pricings['Offer Date'] = pd.to_datetime(ipo_2025_pricings['Offer Date'])
ipo_2025_pricings['Return'] = ipo_2025_pricings['Return'].str.replace(r'[%,]', '', regex=True).astype(float)

ipo_2025_pricings.head(1)

,Company,Symbol,Industry,Offer Date,Shares (millions),Offer Price,1st Day Close,Current Price,Return,SCOOP Rating
0,Vine Hill Capital Investment Corp. II,VHCPU,Blank Check,2025-12-18,20.0,$10.00,$0.00,$0.00,0.0,S/O


In [14]:
# Filter the list to keep only those IPOs with an 'Offer Date' before **1 September 2025**. 
# Also, exclude entries with a 0% return to ensure active tickers are processed. 
# You should see 148 stocks.

ipo_2025_pricings_filtered = ipo_2025_pricings[(ipo_2025_pricings['Offer Date'] < datetime(2025, 9, 1)) & (ipo_2025_pricings['Return'] != 0)]
len(ipo_2025_pricings_filtered)

146

In [15]:
# 3.  **Data Download:** Use `yfinance` to download daily stock data for the filtered tickers and save it to the `stocks_df` dataframe. 
# Make sure you can correctly process the cases when stocks are not present in Yahoo Finance (probably delisted). 
# At this stage you should see about 134 stocks.

import numpy as np
import time
import yfinance as yf

frames = []

for ticker in ipo_2025_pricings_filtered['Symbol']:
    # Work with stock prices
    
    ticker_obj = yf.Ticker(ticker)

    try:
        historyPrices = ticker_obj.history(
            period = "max",
            interval = "1d"
        )
    except Exception as e:
        print(f"{ticker}: {e}")
        continue


    if historyPrices.empty:
        continue

    # generate features for historical prices, and what we want to predict
    historyPrices['Ticker'] = ticker
    historyPrices['Year']= historyPrices.index.year
    historyPrices['Month'] = historyPrices.index.month
    historyPrices['Weekday'] = historyPrices.index.weekday
    historyPrices['Date'] = historyPrices.index.tz_localize(None).normalize()

    # sleep 1 sec between downloads - not to overload the API server
    time.sleep(1)

    # 4.  **Feature Engineering:**
    #     *   Define `growth_252d` as `Close / Close.shift(252)` to represent growth after approximately one year of trading.
    #     *   Calculate **annualized volatility** using the specific formula: `stocks_df['volatility'] = stocks_df['Close'].rolling(30).std() * np.sqrt(252)`.
    historyPrices['growth_252d'] = historyPrices['Close'] / historyPrices['Close'].shift(252)
    historyPrices['volatility'] = historyPrices['Close'].rolling(30).std() * np.sqrt(252)

    frames.append(historyPrices)

stocks_df = pd.concat(frames, ignore_index=True)

len(stocks_df['Ticker'].unique())

HTTP Error 404: {"quoteSummary":{"result":null,"error":{"code":"Not Found","description":"Quote not found for symbol: MJID"}}}
$MJID: possibly delisted; no timezone found
HTTP Error 404: {"quoteSummary":{"result":null,"error":{"code":"Not Found","description":"Quote not found for symbol: EMPG"}}}
$EMPG: possibly delisted; no timezone found
$CAEP: possibly delisted; no timezone found
$PTNM: possibly delisted; no timezone found
$AHL: possibly delisted; no timezone found
$CEPT: possibly delisted; no timezone found
$SDM: possibly delisted; no timezone found
$TBH: possibly delisted; no timezone found
$WGRX: possibly delisted; no timezone found
$AGH: possibly delisted; no timezone found
$EPWK: possibly delisted; no timezone found
$MTSR: possibly delisted; no timezone found
$SKBL: possibly delisted; no timezone found
$MCTR: possibly delisted; no timezone found


132

In [16]:
# **Sharpe Ratio Calculation:** 
# Calculate the Sharpe ratio assuming a risk-free rate of **5.0%** (0.05) - it is close to the current value of the US 10Y Treasury bond yield:
# `stocks_df['Sharpe'] = (stocks_df['growth_252d'] - 0.05) / stocks_df['volatility']`

stocks_df['Sharpe'] = (stocks_df['growth_252d'] - 0.05) / stocks_df['volatility']

In [17]:
stocks_df.tail(2)

,Open,High,Low,Close,Volume,Dividends,Stock Splits,Ticker,Year,Month,Weekday,Date,growth_252d,volatility,Capital Gains,Sharpe
68081,2.90,3.00,2.89,2.95,9533500,0.0,0.0,INLF,2026,9,3,2026-09-24,0.001147,11.880555,NaN,-0.004112
68082,6.25,6.87,4.57,5.10,49266900,0.0,0.0,INLF,2026,9,4,2026-09-25,0.001953,12.155549,NaN,-0.003953


In [18]:
# 6.  **Final Analysis:** Filter the resulting DataFrame to keep data only for the trading day **'2026-09-11'** 
# and compute descriptive statistics (using the `describe()` function).

stocks_df_2026_09_11_df = stocks_df[stocks_df['Date'] == '2026-09-11'][['Ticker', 'growth_252d', 'volatility', 'Sharpe']]
stocks_df_2026_09_11_df.describe()

/home/alexis/projects/stock-market-analytics-zoomcamp/.venv/lib/python3.14/site-packages/pandas/core/nanops.py:1028: RuntimeWarning: invalid value encountered in subtract
  sqr = _ensure_numeric((avg - values) ** 2)


,growth_252d,volatility,Sharpe
count,130.000000,131.000000,130.000000
mean,1.057613,27.516340,inf
std,3.049007,66.879183,NaN
min,0.001005,0.000000,-0.040147
25%,0.141035,2.301309,0.012042
50%,0.594534,7.885748,0.050142
75%,1.038821,23.452319,0.133619
max,33.638270,627.698106,inf


In [19]:
# Expected Observations:
# Compare the median vs. mean for `growth_252d` to see if high-growth outliers are biasing the average.

# Mean: 1.058 -> +5.8% -> they barely growth as a group.
# Median: 0.595 -> -40.5% -> the typical stock lost about 40%.

# We can identify an extreme winner: MGRT

stocks_df_2026_09_11_df.sort_values(by=['growth_252d'], ascending=False).head(5)

,Ticker,growth_252d,volatility,Sharpe
18739,MGRT,33.638270,131.115959,2.561722e-01
23619,MAMK,6.532663,0.000000,inf
5892,BUUU,5.930931,69.792390,8.426321e-02
7301,MAGH,5.451613,0.000001,4.195710e+06
22079,ALM,3.480899,36.399039,9.425795e-02


In [20]:
# *   Identify the count of stocks that successfully reached the 252-day trading milestone.
stocks_df_2026_09_11_df[stocks_df_2026_09_11_df['growth_252d'].isna()]

,Ticker,growth_252d,volatility,Sharpe
2160,NUTR,NaN,0.0,NaN
31519,CIGL,NaN,NaN,NaN


In [21]:
# *   Do you see examples of stocks with risk-adjusted returns (Sharpe ratio) more attractive than pure growth statistics?
stocks_df_2026_09_11_df = stocks_df_2026_09_11_df[stocks_df_2026_09_11_df['growth_252d'].notna()]
stocks_df_2026_09_11_df = stocks_df_2026_09_11_df.sort_values(by=['growth_252d'], ascending=False)
stocks_df_2026_09_11_df['growth_rank'] = np.arange(start=1, stop=len(stocks_df_2026_09_11_df) + 1)
stocks_df_2026_09_11_df.sort_values(by=['Sharpe'], ascending=False).head(10)

# HCMAU, CEPF, TMDE, TLIH

,Ticker,growth_252d,volatility,Sharpe,growth_rank
23619,MAMK,6.532663,0.000000,inf,2
13460,EFTY,2.106592,0.000000,inf,8
7301,MAGH,5.451613,0.000001,4.195710e+06,4
14893,HCMAU,1.034585,0.182827,5.385326e+00,34
1351,CEPF,1.024233,0.351583,2.770994e+00,36
32241,TMDE,0.776166,0.414673,1.751178e+00,54
23310,TLIH,0.989673,0.648547,1.448889e+00,39
67207,CEPO,1.031160,0.795258,1.233763e+00,35
33329,HXHX,0.409565,0.477354,7.532461e-01,84
59630,FBGL,0.576923,0.838018,6.287732e-01,67


#### What is the median Sharpe ratio for these stocks as of September 11, 2026?

**From the table obtained after #6, we can see that the median Sharpe ratio is 0.0501.**

### Question 3: [IPO] 'Fixed Months Holding Strategy'

**What is the optimal number of months (1 to 12) to hold a newly IPO'd stock in order to maximize the median growth value?**

#### Goal:
Investigate the performance of 2025 IPO stocks over fixed time horizons (1 to 12 months) to identify the holding period that yields the highest median return relative to the first day's closing price.

#### Steps:
1. **Data Source:** Use the existing `stocks_df` containing daily OHLCV data and calculated features for the 2025 IPOs filtered in Question 2.
2. **Feature Engineering:** Calculate 12 future growth columns representing fixed holding periods:
   - `future_growth_1_m`, `future_growth_2_m`, ..., `future_growth_12_m`.
   - Assume 1 month equals 21 trading days (e.g., 1 month = 21 days, 2 months = 42 days, ..., 12 months = 252 days).
3. **Identify Entry Points:** For each ticker, determine the first available trading day (`min_date`) and the corresponding closing price.
4. **Data Alignment:** Perform an inner join between the `min_date` records and the full growth dataset. This isolates the returns for each stock starting specifically from its IPO date.
5. **Median Analysis:** Compute descriptive statistics for these 12 columns. Specifically, identify the month where the **50th percentile (median)** value is highest.

#### Observations:
- Do you observe any trend of the median growth over time? What does it mean for an investor eager to base their strategy on the newly IPOed companies?
- Compare the median to the mean to understand how extreme outliers (high-growth stocks) affect the average versus the typical stock's performance.

**Answer:** Based on the analysis of the filtered tickers, what is the optimal holding period (in months) and its corresponding maximum median growth value?

In [22]:
# 2. **Feature Engineering:** Calculate 12 future growth columns representing fixed holding periods:
#    - `future_growth_1_m`, `future_growth_2_m`, ..., `future_growth_12_m`.
#    - Assume 1 month equals 21 trading days (e.g., 1 month = 21 days, 2 months = 42 days, ..., 12 months = 252 days).
growth_cols = [f'future_growth_{m}_m' for m in range(1, 13)]

for m in range(1, 13):
    stocks_df[f'future_growth_{m}_m'] = (
        stocks_df.groupby('Ticker')['Close'].shift(-21 * m) / stocks_df['Close']
    )


In [23]:
# 3. **Identify Entry Points:** For each ticker, determine the first available trading day (`min_date`) and the corresponding closing price.
min_dates = (
    stocks_df.groupby('Ticker')['Date'].min()
    .reset_index()
    .rename(columns={'Date': 'min_date'})
)

# 4. **Data Alignment:** Perform an inner join between the `min_date` records and the full growth dataset. This isolates the returns for each stock starting specifically from its IPO date.
growth_cols = [f'future_growth_{m}_m' for m in range(1, 13)]
first_day_df = stocks_df.merge(
    min_dates, left_on=['Ticker', 'Date'], right_on=['Ticker', 'min_date'], how='inner'
)
len(first_day_df)


132

In [24]:
# 5. **Median Analysis:** Compute descriptive statistics for these 12 columns. Specifically, identify the month where the **50th percentile (median)** value is highest.
stats = first_day_df[growth_cols].describe().T
stats

,count,mean,std,min,25%,50%,75%,max
future_growth_1_m,132.0,1.119635,1.254616,0.072941,0.708835,0.935351,1.098038,9.466019
future_growth_2_m,131.0,1.294861,1.957999,0.106197,0.584863,0.890411,1.152849,16.066783
future_growth_3_m,131.0,1.252014,1.912626,0.090147,0.442477,0.827160,1.149994,16.344464
future_growth_4_m,131.0,1.135939,1.558464,0.063927,0.392759,0.741678,1.203576,11.386643
future_growth_5_m,131.0,1.084528,1.468733,0.032245,0.400704,0.667857,1.083509,9.495146
future_growth_6_m,131.0,1.114789,1.668273,0.022075,0.308026,0.726000,1.056403,9.894552
future_growth_7_m,131.0,1.146371,2.278159,0.013229,0.245548,0.662000,1.019297,20.741478
future_growth_8_m,131.0,1.004473,2.046907,0.015130,0.200983,0.603508,1.036588,21.370826
future_growth_9_m,131.0,1.158480,2.999642,0.014675,0.174587,0.580294,1.031006,27.627298
future_growth_10_m,131.0,1.185497,3.422082,0.011947,0.174333,0.533333,1.034765,34.687666


In [25]:
stats['50%'].idxmax(), stats['50%'].max()

('future_growth_1_m', np.float64(0.9353509333873746))

### Question 4: [Strategy] Simple RSI-Based Trading Strategy

**What is the total profit (in $ thousands) you would have earned by investing $1000 every time a stock was oversold (RSI < 30)?**

#### Goal:
Apply a simple rule-based trading strategy using the Relative Strength Index (RSI) technical indicator to identify oversold signals and calculate cumulative profits over a 25-year period.

#### Steps:
1.  **Data Acquisition:** Use the provided brotli-compressed parquet file containing precomputed technical and macro indicators for a broad set of tickers.
    ```python
    import gdown
    import pandas as pd
    file_id = "1grCTCzMZKY5sJRtdbLVCXg8JXA8VPyg-"
    gdown.download(f"https://drive.google.com/uc?id={file_id}", "data.parquet", quiet=False)
    df = pd.read_parquet("data.parquet", engine="pyarrow")
    ```
2.  **Strategy Setup:** Define the RSI threshold for an "oversold" signal as **RSI < 30**.
3.  **Filtering:** Filter the dataset to isolate trades occurring between **2000-01-01** and **2025-06-01** where the RSI signal was triggered.
4.  **Profit Calculation:**
    - Assume an investment of **$1,000** for every signal.
    - Use the **30-day forward return** (`growth_future_30d`) to determine the outcome of each trade.
    - Calculate Net Income: `net_income = 1000 * (selected_df['growth_future_30d'] - 1).sum()`

#### Observations:
- Increasing the threshold from 25 (we used last year) to 30 significantly increases the number of trading opportunities (from ~1,568 to 5,206).
- With an average 30-day return of **1.26%** and a win rate of **55.13%**, the strategy shows consistent, albeit modest, capital growth over the long term.

**Answer:** Based on a few thousands of trades, what is the net income earned (in $ thousands)?

In [26]:
import gdown

file_id = "1grCTCzMZKY5sJRtdbLVCXg8JXA8VPyg-"
gdown.download(f"https://drive.google.com/uc?id={file_id}", "data.parquet", quiet=False)
df = pd.read_parquet("data.parquet", engine="pyarrow")

Downloading...
From (original): https://drive.google.com/uc?id=1grCTCzMZKY5sJRtdbLVCXg8JXA8VPyg-
From (redirected): https://drive.google.com/uc?id=1grCTCzMZKY5sJRtdbLVCXg8JXA8VPyg-&confirm=t&uuid=88a241ad-d8a5-417f-ad8b-25ed2c75f222
To: /home/alexis/projects/stock-market-analytics-zoomcamp/src/stock_market_analytics_zoomcamp/homeworks/data.parquet
100%|██████████| 130M/130M [00:03<00:00, 34.3MB/s] 


In [27]:
# 2-3. Oversold signal (RSI < 30) between 2000-01-01 and 2025-06-01
rsi_threshold = 30
selected_df = df[
    (df['rsi'] < rsi_threshold)
    & (df['Date'] >= '2000-01-01')
    & (df['Date'] <= '2025-06-01')
]
len(selected_df)  

5206

In [28]:
# 4. $1000 per signal, held for 30 days
net_income = 1000 * (selected_df['growth_future_30d'] - 1).sum()
net_income  

np.float64(65805.58960747933)

In [29]:
# Observations
(selected_df['growth_future_30d'] - 1).mean(), (selected_df['growth_future_30d'] > 1).mean()   # 0.0126 -> +1.26% average per trade, 0.5513 -> 55.13% win rate

(np.float64(0.01264033607519772), np.float64(0.5512869765655013))

### Q5. [Exploratory, Optional] Predicting a Positive-Return IPO

Most of the strategies for investing in IPOs deliver **negative average and median returns** (and even the 75th percentile).

**Question:**
How would you change the strategy if you want to **increase profitability**?

> This is an open-ended brainstorming question — propose ideas for identifying IPOs with positive future returns or building a more effective trading strategy.